#### **볼린저 밴드 투자 전략 백테스팅**

- 이동 평균선 생성: 데이터 20개의 평균값
- 상단 밴드 생성: 이동 평균선 + ( 2 * 20개 데이터의 표준편차 )
- 하단 밴드 생성: 이동 평균선 - ( 2 * 20개 데이터의 표준편차 )
- 가격이 하단 밴드보다 낮은 경우 매수
- 가격이 상단 밴드보다 높은 경우 매도

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import datetime

In [2]:
# AMZN (아마존) 데이터를 로드

df = pd.read_csv('../csv/AMZN.csv', index_col = 'Date')
df.head()

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
1997-05-15,2.437500,2.500000,1.927083,1.958333,1.958333,72156000
1997-05-16,1.968750,1.979167,1.708333,1.729167,1.729167,14700000
1997-05-19,1.760417,1.770833,1.625000,1.708333,1.708333,6106800
1997-05-20,1.729167,1.750000,1.635417,1.635417,1.635417,5467200
1997-05-21,1.635417,1.645833,1.375000,1.427083,1.427083,18853200


In [4]:
# 결측치 데이터가 존재하는가?

df.info()

<class 'pandas.DataFrame'>
Index: 5563 entries, 1997-05-15 to 2019-06-24
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Open       5563 non-null   float64
 1   High       5563 non-null   float64
 2   Low        5563 non-null   float64
 3   Close      5563 non-null   float64
 4   Adj Close  5563 non-null   float64
 5   Volume     5563 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 304.2+ KB


In [6]:
# 결측치나 무한대 데이터를 제거

flag = df.isin( [np.nan, np.inf, -np.inf] ).any(axis = 1)

In [7]:
# flag의 부정을 해서 데이터 필터링

df = df.loc[~flag, ]

In [9]:
len(df)

5563

In [10]:
# 종가 이외의 데이터를 제외
# 이동평균선 → rolling

df = df[['Adj Close']]
df['center'] = df['Adj Close'].rolling(20).mean()

In [12]:
df.iloc[18:24, ]

,Adj Close,center
Date,,
1997-06-11,1.541667,NaN
1997-06-12,1.604167,1.574740
1997-06-13,1.583333,1.555990
1997-06-16,1.572917,1.548177
1997-06-17,1.505208,1.538021
1997-06-18,1.510417,1.531771


In [13]:
# 상단 밴드, 하단 밴드 생성

std_value = 2 * df['Adj Close'].rolling(20).std()

df['ub'] = df['center'] + std_value
df['lb'] = df['center'] - std_value

In [14]:
df.iloc[19:25]

,Adj Close,center,ub,lb
Date,,,,
1997-06-12,1.604167,1.574740,1.836333,1.313146
1997-06-13,1.583333,1.555990,1.745696,1.366283
1997-06-16,1.572917,1.548177,1.719869,1.376485
1997-06-17,1.505208,1.538021,1.693045,1.382996
1997-06-18,1.510417,1.531771,1.680201,1.383341
1997-06-19,1.510417,1.535938,1.676462,1.395413


In [ ]:
# index 값을 시계열로 변경
df.index = pd.to_datetime(df.index)

In [17]:
# 투자 시작 시간 설정
start = '2010-01-01'

test_df = df.loc[start: , ]
test_df.head()

,Adj Close,center,ub,lb
Date,,,,
2010-01-04,133.899994,133.984001,141.460445,126.507556
2010-01-05,134.690002,133.839500,141.132776,126.546225
2010-01-06,132.250000,133.741500,141.066419,126.416581
2010-01-07,130.000000,133.536000,141.045671,126.026329
2010-01-08,133.520004,133.646500,141.082939,126.210062


In [23]:
# 구매 상태를 입력할 수 있는 공간 생성

test_df['trade'] = ''
test_df.head()

,Adj Close,center,ub,lb,trade
Date,,,,,
2010-01-04,133.899994,133.984001,141.460445,126.507556,
2010-01-05,134.690002,133.839500,141.132776,126.546225,
2010-01-06,132.250000,133.741500,141.066419,126.416581,
2010-01-07,130.000000,133.536000,141.045671,126.026329,
2010-01-08,133.520004,133.646500,141.082939,126.210062,


##### **보유 내역 추가**

- 조건식
    - 상단 밴드보다 수정 주가가 높거나 같은 경우
        - 내가 현재 보유 중이라면? (전일의 trade가 `buy`라면 )
            - **매도** (`trade = ''`)
        - 보유 중이 아니라면?
            - **유지** (`trade = ''`)

    <br>

    - 하단 밴드보다 수정 주가가 작거나 같은 경우
        - 내가 현재 보유 중이라면?
            - **유지** (`trade = 'buy'`)
        - 보유 중이 아니라면?
            - **매수** (`trade = 'buy'`)
    
    <br>        
    
    - 수정 주가가 밴드 사이에 존재하는 경우
        - 내가 현재 보유 중이라면?
            - **유지** (`trade = 'buy'`)
        - 보유 중이 아니라면?
            - **유지** (`trade = ''`)

In [39]:
for i in test_df.index:

    # 상단 밴드보다 수정 종가가 높은 경우
    if test_df.loc[i, 'Adj Close'] >= test_df.loc[i, 'ub']:
        # 보유 중이라면 trade = '', 아니라면 trade = ''
        test_df.loc[i, 'trade'] = ''

    # 하단 밴드보다 수정 종가가 낮은 경우
    elif test_df.loc[i, 'Adj Close'] <= test_df.loc[i, 'lb']:
        # 보유 중이라면 trade = 'buy', 아니라면 trade = 'buy'
        test_df.loc[i, 'trade'] = 'buy'
    
    else:
        # 보유 중인 경우 trade = 'buy', 보유 중이 아니면 trade = ''
        # 전날의 trade가 buy인 경우: 보유 중
        if test_df.shift().loc[i, 'trade'] == 'buy':
            test_df.loc[i, 'trade'] = 'buy'
        else:
            test_df.loc[i, 'trade'] = ''

In [40]:
test_df['trade'].value_counts()

trade
       1521
buy     863
Name: count, dtype: int64

##### 수익률 계산

- 매수한 날의 수정주가와 매도한 날의 수정 주가를 이용하여 수익률 계산
- 매수한 날의 수정 주가
    - 전날의 trade = ''이고 오늘의 trade = 'buy'인 날짜의 수정 종가
- 매도한 날의 수정 주가
    - 전날의 trade = 'buy'이고 오늘의 trade = ''인 날짜의 수정 종가
- 수익률
    - 매도날의 수정종가 / 매수날의 수정종가

In [45]:
# 수익률 column 생성, 1로 채워준다

test_df['rtn'] = 1.0

In [46]:
test_df.head()

,Adj Close,center,ub,lb,trade,rtn
Date,,,,,,
2010-01-04,133.899994,133.984001,141.460445,126.507556,,1.0
2010-01-05,134.690002,133.839500,141.132776,126.546225,,1.0
2010-01-06,132.250000,133.741500,141.066419,126.416581,,1.0
2010-01-07,130.000000,133.536000,141.045671,126.026329,,1.0
2010-01-08,133.520004,133.646500,141.082939,126.210062,,1.0


In [ ]:
for i in test_df.index:
    # 매수 가격 형성
    if (test_df.shift().loc[i, 'trade'] == '') & (test_df.loc[i, 'trade'] == 'buy'):
        buy = test_df.loc[i, 'Adj Close']
        print(f'매수일: {i}, 매수가: {buy}')

    # 매도 가격 형성
    elif (test_df.shift().loc[i, 'trade'] == 'buy') & (test_df.loc[i, 'trade'] == ''):
        sell = test_df.loc[i, 'Adj Close']

        # 수익률 계산
        rtn = sell / buy
        test_df.loc[i, 'rtn'] = rtn
        print(f'매도일: {i}, 매도가: {sell}, 수익률: {rtn}')

매수일: 2010-05-04 00:00:00, 매수가: 129.830002
매도일: 2010-08-04 00:00:00, 매도가: 127.580002, 수익률: 0.9826696451872502
매수일: 2010-11-16 00:00:00, 매수가: 157.779999
매도일: 2010-11-24 00:00:00, 매도가: 177.25, 수익률: 1.1233996775472155
매수일: 2011-01-21 00:00:00, 매수가: 177.419998
매도일: 2011-03-30 00:00:00, 매도가: 179.419998, 수익률: 1.0112726864082142
매수일: 2011-06-06 00:00:00, 매수가: 185.690002
매도일: 2011-06-27 00:00:00, 매도가: 201.25, 수익률: 1.0837955615941024
매수일: 2011-08-04 00:00:00, 매수가: 201.479996
매도일: 2011-10-14 00:00:00, 매도가: 246.710007, 수익률: 1.2244888420585436
매수일: 2011-10-26 00:00:00, 매수가: 198.399994
매도일: 2012-01-18 00:00:00, 매도가: 189.440002, 수익률: 0.9548387486342363
매수일: 2012-10-10 00:00:00, 매수가: 244.990005
매도일: 2012-11-29 00:00:00, 매도가: 251.270004, 수익률: 1.0256336947297096
매수일: 2013-04-29 00:00:00, 매수가: 249.740005
매도일: 2013-06-07 00:00:00, 매도가: 276.869995, 수익률: 1.1086329360808655
매수일: 2013-08-14 00:00:00, 매수가: 291.339996
매도일: 2013-09-18 00:00:00, 매도가: 312.029999, 수익률: 1.0710166928127507
매수일: 2014-01-24 00:00:00, 매

In [53]:
# 누적 수익률 계산 → rtn 누적곱

acc_rtn = 1.0

for i in test_df.index:
    rtn = test_df.loc[i, 'rtn']
    acc_rtn *= rtn

acc_rtn

np.float64(3.138061358619031)

In [54]:
test_df['acc_rtn'] = test_df['rtn'].cumprod()

In [55]:
test_df.iloc[-1, -1]

np.float64(3.138061358619031)

In [56]:
# buyandhold
# 투자 기간 첫날 → 매수가
# 투자 마지막 날 → 매도가

bnh_rtn = test_df.iloc[-1, 0] / test_df.iloc[0, 0]
print(bnh_rtn)

14.249095911087196


#### 볼린저 밴드 함수화

1. **밴드를 생성하는 함수**
    - 매개변수
        - 데이터, 기준 컬럼(Adj Close), 투자의 시작 시간(2010-01-01), 종료 시간(현재 시간), 묶음 데이터 개수(20)
        - _df, _col, _start, _end, _cnt
    - _df 깊은 복사
    - 인덱스가 Date가 아니라면 인덱스를 Date로 변경
        - 컬럼들 중 Date가 존재한다면? → index가 Date가 아니다
    - 인덱스를 시계열 변환
    - 시계열 데이터에서 time_zone 제거
    - 결측치, 무한대 데이터 제외
    - 기준이 되는 컬럼을 제외하고는 모두 제거
    - 이동 평균선, 상단 밴드, 하단 밴드 생성 (파생변수 생성)
    - 시작시간과 종료 시간으로 데이터를 필터링
    - 위 결과를 되돌려준다.

In [94]:
def create_band(
        _df,
        _col = 'Adj Close',
        _start = '2010-01-01',
        _end = datetime.strftime(pd.to_datetime(datetime.now()), format='%Y-%m-%d'),
        _cnt = 20
):
    # _df 깊은 복사
    df = _df.copy()

    # 인덱스가 Date가 아니라면 인덱스를 Date로 변경
    if df.columns.isin(['Date']).any():
        df.index = df['Date']
    else:
        pass

    # 시계열
    df.index = pd.to_datetime(df.index)

    # time_zone 제거
    df.index = df.index.tz_localize(None)

    # 결측치, 무한대 데이터 제외
    flag = df.isin( [np.nan, np.inf, -np.inf] ).any(axis = 1)
    df = df.loc[~flag, ]

    # 기준 컬럼 제외 모두 제거
    df = df[[_col]]
    
    # 이평선
    df['center'] = df[_col].rolling(_cnt).mean()

    # 상단/하단 밴드
    std_value = 2 * df[_col].rolling(_cnt).std()
    df['ub'] = df['center'] + std_value
    df['lb'] = df['center'] - std_value

    # 시작 시간과 종료 시간으로 데이터 필터링
    df = df.loc[_start:_end, ]

    # return
    return df

In [95]:
df2 = pd.read_csv('../csv/aapl.csv')

create_band(df2)

,Adj Close,center,ub,lb
Date,,,,
2010-01-04,26.782711,25.037723,27.046734,23.028713
2010-01-05,26.829010,25.169503,27.288098,23.050908
2010-01-06,26.402260,25.307290,27.366449,23.248130
2010-01-07,26.353460,25.436879,27.410937,23.462821
2010-01-08,26.528664,25.525609,27.529742,23.521475
...,...,...,...,...
2019-06-18,198.449997,185.432500,201.032574,169.832427
2019-06-19,197.869995,185.996000,202.558154,169.433846
2019-06-20,199.460007,186.830000,204.361771,169.298229


In [ ]:
# 강사님 풀이
# start나 end를 바꾸고 싶다면 문자로 통일하지 말고 시계열로 통일하라

def create_band2(
        _df,
        _col = 'Adj Close',
        _start = '2010-01-01',
        _end = datetime.now(),
        _cnt = 20
):
    
    df = _df.copy()

    if 'Date' in df.columns:
        df.set_index('Date', inplace=True)
    
    df.index = pd.to_datetime(df.index)

    df.index = df.index.tz_localize(None)

    flag = df.isin( [ np.nan, np.inf, -np.inf ] ).any(axis=1)
    df = df.loc[~flag, ]

    df = df[[_col]]

    df['center'] = df[_col].rolling(_cnt).mean()

    std_value = 2 * df[_col].rolling(_cnt).std()
    df['ub'] = df['center'] + std_value
    df['lb'] = df['center'] - std_value

    df = df.loc[_start:_end, ]

    return df

In [106]:
df2 = pd.read_csv('../csv/aapl.csv')

band_df = create_band2(df2)

2. **보유내역을 생성하는 함수**
    - 매개변수
        - 밴드가 생성된 데이터프레임
        - 기준 컬럼 (option)
    - trade column을 생성하여 '' 값으로 채워준다.
    - 밴드의 값들과 기준이 되는 컬럼의 값을 이용하여 보유 내역 생성
    - 결과 return

In [107]:
def create_trade(_df):
    # 기준 컬럼 이름을 어떻게 알 것인가?: 첫번째 함수의 return으로 나온 df의 첫번째 컬럼만
    col = _df.columns[0]
    df = _df.copy()
    df['trade'] = ''

    for i in df.index:
        if df.loc[i, col] >= df.loc[i, 'ub']:
            # 매도
            df.loc[i, 'trade'] = ''
        elif df.loc[i, col] <= df.loc[i, 'lb']:
            # 매수
            df.loc[i, 'trade'] = 'buy'
        else:
            if df.shift().loc[i, 'trade'] == 'buy':
                df.loc[i, 'trade'] = 'buy'
            else:
                df.loc[i, 'trade'] = ''
    
    return df

In [108]:
trade_df = create_trade(band_df)

trade_df['trade'].value_counts()

trade
       1439
buy     945
Name: count, dtype: int64

3. **수익률 계산 함수**
    - 매개변수
        - 두번째 함수의 결과를 받아오는 데이터 매개변수
    - 기준이 되는 column의 이름을 변수에 저장
    - df에 rtn column을 생성해서 1로 채워준다.
    - 매수날과 매도날을 조건식을 생성하여 가격을 생성하고 수익률 계산 뒤 rtn column에 대입
    - 누적 수익률 컬럼을 생성하여 대입
    - 생성된 데이터 프레임과 최종 누적 수익률을 되돌려준다.

In [141]:
def create_rtn(_df):
    col = _df.columns[0]
    df = _df.copy()
    
    df['rtn'] = 1.0

    # 수익률 계산
    for i in df.index:
        # 매수
        if (df.shift().loc[i, 'trade'] == '') & (df.loc[i, 'trade'] == 'buy'):
            buy = df.loc[i, col]
            print(f"매수일: {i}, 매수가: {buy}")
            print()
        elif (df.shift().loc[i, 'trade'] == 'buy') & (df.loc[i, 'trade'] == ''):
            sell = df.loc[i, col]
            rtn = sell / buy
            df.loc[i, 'rtn'] = rtn
            print(f"매도일: {i}, 매도가: {sell}, 수익률: {rtn}")
            print()
    # 누적 수익률
    df['acc_rtn'] = df['rtn'].cumprod()
    # 최종 수익률
    acc_rtn = df.iloc[-1, -1]

    return df, acc_rtn

In [142]:
rtn_df, acc_rtn = create_rtn(trade_df)

매수일: 2010-01-22 00:00:00, 매수가: 24.747818

매도일: 2010-03-01 00:00:00, 매도가: 26.154476, 수익률: 1.0568396777445188

매수일: 2010-08-24 00:00:00, 매수가: 30.026524

매도일: 2010-09-08 00:00:00, 매도가: 32.90366, 수익률: 1.0958198158401553

매수일: 2011-03-16 00:00:00, 매수가: 41.299767

매도일: 2011-07-01 00:00:00, 매도가: 42.957966, 수익률: 1.040150323366231

매수일: 2011-11-14 00:00:00, 매수가: 47.463268

매도일: 2011-12-27 00:00:00, 매도가: 50.876015, 수익률: 1.0719029081604747

매수일: 2012-04-16 00:00:00, 매수가: 72.601524

매도일: 2012-06-18 00:00:00, 매도가: 73.308609, 수익률: 1.0097392583659814

매수일: 2012-10-08 00:00:00, 매수가: 80.207954

매도일: 2013-07-29 00:00:00, 매도가: 57.243137, 수익률: 0.7136840443530077

매수일: 2013-09-11 00:00:00, 매수가: 60.184383

매도일: 2013-10-18 00:00:00, 매도가: 65.48336, 수익률: 1.0880457144505413

매수일: 2014-01-03 00:00:00, 매수가: 70.019096

매도일: 2014-03-25 00:00:00, 매도가: 70.960335, 수익률: 1.0134426042861222

매수일: 2014-10-15 00:00:00, 매수가: 89.842468

매도일: 2014-10-23 00:00:00, 매도가: 96.557182, 수익률: 1.0747387527243797

매수일: 2014-12-12 00:00:

In [143]:
acc_rtn

np.float64(1.3923287814461949)

In [144]:
rtn_df

,Adj Close,center,ub,lb,trade,rtn,acc_rtn
Date,,,,,,,
2010-01-04,26.782711,25.037723,27.046734,23.028713,,1.0,1.000000
2010-01-05,26.829010,25.169503,27.288098,23.050908,,1.0,1.000000
2010-01-06,26.402260,25.307290,27.366449,23.248130,,1.0,1.000000
2010-01-07,26.353460,25.436879,27.410937,23.462821,,1.0,1.000000
2010-01-08,26.528664,25.525609,27.529742,23.521475,,1.0,1.000000
...,...,...,...,...,...,...,...
2019-06-18,198.449997,185.432500,201.032574,169.832427,buy,1.0,1.392329
2019-06-19,197.869995,185.996000,202.558154,169.433846,buy,1.0,1.392329
2019-06-20,199.460007,186.830000,204.361771,169.298229,buy,1.0,1.392329


In [145]:
class Investing():
    def __init__(self, _df, _col = 'Adj Close', _start = '2010-01-01', _end = datetime.now()):
        self.df = _df
        self.col = _col
        self.start = _start
        self.end = _end
    
    # 바이앤홀드 함수
    def bnh(self):
        df = self.df.copy()
        if 'Date' in df.columns:
            df.set_index('Date', inplace=True)
        df.index = pd.to_datetime(df.index)
        df = df.loc[self.start:self.end, [self.col]]
        buy = df.iloc[0, 0]
        sell = df.iloc[-1, 0]
        return sell / buy
    
    # 볼린져 밴드 함수
    def boll(self, _cnt = 20):
        band_df = create_band(self.df, self.col, self.start, self.end, _cnt)
        trade_df = create_trade(band_df)
        rtn_df, acc_rtn = create_rtn(trade_df)
        return rtn_df, acc_rtn

In [146]:
df3 = pd.read_csv('../csv/MSFT.csv')

In [147]:
invest = Investing(df3)

In [148]:
invest.bnh()

np.float64(5.6387313298309785)